In [28]:
##### Compiles data for just the US

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats
from rasterio.mask import mask
from shapely.geometry import box
from functools import reduce

In [11]:
# Get the current working directory
cd = Path.cwd().parent 

In [15]:
##### Geographies

input = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")
output = f"{cd}/Data/Misc./COMPS/USA_subnational_geography.shp"

input = input[input['ISO3'] == 'USA']
input.to_csv(output, index=False)

In [17]:
##### Capital intensities data 

input = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_capital_intensity.csv")
output = f"{cd}/Data/Misc./COMPS/USA_subnational_capital_intensities.csv"

input['ISO3'] = input['PROJ_ID'].str[:3]

input = input[input['ISO3'] == 'USA']
input.to_csv(output, index=False)

In [18]:
##### Labor intensities data 

input = pd.read_csv(f"{cd}/Data/Clean/Intensities/subnational_labor_intensity.csv")
output = f"{cd}/Data/Misc./COMPS/USA_subnational_labor_intensities.csv"

input['ISO3'] = input['PROJ_ID'].str[:3]

input = input[input['ISO3'] == 'USA']
input.to_csv(output, index=False)

In [20]:
##### County intensities data 

input = pd.read_csv(f"{cd}/Data/Clean/Intensities/country_intensities.csv")
output = f"{cd}/Data/Misc./COMPS/USA_intensities.csv"

input = input[input['ISO3'] == 'USA']
input.to_csv(output, index=False)

In [24]:
##### Predictor data (vectors)

def filter_usa_vectors(cd):
    input_dir = f"{cd}/Data/Clean/Predictors/Vectors"
    output_dir = f"{cd}/Data/Misc./COMPS/Predictors/Vector"
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if filename.endswith(".csv"):
            df = pd.read_csv(os.path.join(input_dir, filename))
            df['ISO3'] = df['PROJ_ID'].str[:3]
            df = df[df['ISO3'] == 'USA']
            df.to_csv(os.path.join(output_dir, filename), index=False)

filter_usa_vectors(cd)

In [27]:
##### Predictor data (Rasters)

def clip_usa_rasters(cd):
    input_dir = f"{cd}/Data/Clean/Predictors/Rasters"
    output_dir = f"{cd}/Data/Misc./COMPS/Predictors/Raster"
    os.makedirs(output_dir, exist_ok=True)

    usa_bbox = box(-180, 18, -65, 72)

    for filename in os.listdir(input_dir):
        if filename.endswith(".tif") or filename.endswith(".tiff"):
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)

            with rasterio.open(input_path) as src:
                clipped, transform = mask(src, [usa_bbox], crop=True)
                profile = src.profile.copy()  # copy first
                profile.update(               # then update
                    height=clipped.shape[1],
                    width=clipped.shape[2],
                    transform=transform
                )

            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(clipped)

clip_usa_rasters(cd)

In [40]:
##### Combine into master sub-national dfs

### Predictors  
def merge_vector_csvs(cd):
    vector_dir = f"{cd}/Data/Misc./COMPS/Predictors/Vector"
    
    dfs = [
        pd.read_csv(os.path.join(vector_dir, f)).drop(columns='ISO3')
        for f in os.listdir(vector_dir)
        if f.endswith(".csv")
    ]

    merged = reduce(lambda left, right: pd.merge(left, right, on='PROJ_ID', how='outer'), dfs)
    
    return merged

predictors = merge_vector_csvs(cd)

### Capital 
capital = pd.read_csv(f"{cd}/Data/Misc./COMPS/USA_subnational_capital_intensities.csv")

capital = capital.merge(predictors, on='PROJ_ID', how='outer')

capital['STATE_ID'] = capital['PROJ_ID'].str[4:6]

col_to_keep = ['PROJ_ID', 'STATE_ID', 'capital_intensity_USD_per_USD',
       'pct_cropland_irrigated', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'GDP_pc',
       'USD_production_per_HA', 'cereals_share_production_USD',
       'fibres_share_production_USD', 'fruits_share_production_USD',
       'oilcrops_share_production_USD', 'pulses_share_production_USD',
       'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'other_share_production_USD', 'share_vlarge_field', 'share_large_field',
       'share_medium_field', 'share_small_field', 'share_vsmall_field']

capital = capital[col_to_keep]

capital.to_csv(f"{cd}/Data/Misc./COMPS/USA_CAPITAL_MASTER.csv", index=False)

### Labor 
labor = pd.read_csv(f"{cd}/Data/Misc./COMPS/USA_subnational_labor_intensities.csv")

labor = labor.merge(predictors, on='PROJ_ID', how='outer')

labor['STATE_ID'] = labor['PROJ_ID'].str[4:6]

col_to_keep = ['PROJ_ID', 'STATE_ID', 'labor_intensity_jobs_per_million_USD',
       'pct_cropland_irrigated', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'GDP_pc',
       'USD_production_per_HA', 'cereals_share_production_USD',
       'fibres_share_production_USD', 'fruits_share_production_USD',
       'oilcrops_share_production_USD', 'pulses_share_production_USD',
       'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'other_share_production_USD', 'share_vlarge_field', 'share_large_field',
       'share_medium_field', 'share_small_field', 'share_vsmall_field']

labor = labor[col_to_keep]

labor.to_csv(f"{cd}/Data/Misc./COMPS/USA_LABOR_MASTER.csv", index=False)


In [37]:
labor

Index(['PROJ_ID', 'GEO_ID_NAME', 'ag_jobs', 'total_production_USD',
       'labor_intensity_jobs_per_million_USD', 'ISO3',
       'pct_cropland_irrigated', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'GDP_pc',
       'USD_production_per_HA', 'cereals_share_production_USD',
       'fibres_share_production_USD', 'fruits_share_production_USD',
       'oilcrops_share_production_USD', 'pulses_share_production_USD',
       'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'other_share_production_USD', 'share_vlarge_field', 'share_large_field',
       'share_medium_field', 'share_small_field', 'share_vsmall_field',
       